# ProjectTaxila: Colab/Kaggle reproduction launcher

This small launcher runs the canonical analysis, preserves the executed notebook and output manifest, and writes an execution receipt containing the code revision, environment and evidence-lock result. It does not change the scientific interpretation boundary: outputs are relative field-inspection priorities, not confirmed deterioration or causal climate effects.

## Setup

Open this notebook from a complete ProjectTaxila checkout. In Colab, upload and extract the repository ZIP to `/content` if the private GitHub repository is not directly accessible. In Kaggle, attach the repository/data as an input dataset or copy the checkout to `/kaggle/working`. Never paste a GitHub token into a notebook cell.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

def locate_project_root() -> Path:
    markers = [Path.cwd(), *Path.cwd().parents, Path('/content/ProjectTaxila'), Path('/content/ProjectTaxila-main'), Path('/kaggle/working/ProjectTaxila')]
    for candidate in markers:
        if (candidate / 'scripts/run_reproduction.py').is_file() and (candidate / 'data/Taxila_CHIP_Frozen_Evidence_Data').is_dir():
            return candidate.resolve()
    raise FileNotFoundError('Complete ProjectTaxila checkout not found. Upload/extract it, then rerun this cell.')

PROJECT_ROOT = locate_project_root()
os.chdir(PROJECT_ROOT)
print('Project root:', PROJECT_ROOT)

In [ ]:
# Exact claim-critical environment. Installation may take several minutes on a fresh runtime.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT_ROOT / 'requirements-reproduction.txt')])

## Run and inspect

Use `validation` for a quick engineering run. Use `publication` only for the full 50,000-draw and proxy-model refit; it is CPU-intensive. GPU runtimes do not materially accelerate this pipeline.

In [ ]:
PROFILE = 'validation'  # change to 'publication' for the full scientific rerun
RUN_DIR = PROJECT_ROOT / 'output' / 'colab-kaggle-run' / PROFILE
if RUN_DIR.exists():
    raise FileExistsError(f'{RUN_DIR} already exists; choose a new RUN_DIR to preserve the previous receipt.')
subprocess.check_call([sys.executable, str(PROJECT_ROOT / 'scripts/run_reproduction.py'), '--profile', PROFILE, '--output-dir', str(RUN_DIR)])
receipt = json.loads((RUN_DIR / 'execution_receipt.json').read_text(encoding='utf-8'))
receipt

## Checks

A trustworthy run has `status: PASS`, a passing data-quality report, and all applicable headline evidence locks passing. A publication run that reports `REVIEW` must not be described as a clean reproduction; inspect the failed claims in the receipt and `artifacts/validation/headline_evidence_lock.csv`.